<a href="https://colab.research.google.com/github/i-malur/bootcamp-data-analytics-mentoria-19.06/blob/main/mentoria_19_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# bibliotecas de base
import pandas as pd
import numpy as np

# bibliotecas para visualização dos dados
import seaborn as sns
import matplotlib.pyplot as plt

# biblioteca para estatística
import scipy.stats as stat

In [ ]:
url = 'https://github.com/i-malur/bootcamp-data-analytics-mentoria-19.06/raw/refs/heads/main/populacao_br.xlsx'
df_populacao_uf = pd.read_excel(url, sheet_name='BRASIL E UFs')

In [ ]:
df_populacao_uf.head()

,UF,POPULACAO,Unnamed: 2
0,Brasil,213421037,NaN
1,Norte,18801282,NaN
2,Rondônia,1751950,NaN
3,Acre,884372,NaN
4,Amazonas,4321616,NaN


In [ ]:
# veio uma coluna a mais no dataframe sem valores. Apaguei para deixar mais fácil a visualização
df_populacao_uf = df_populacao_uf.drop('Unnamed: 2', axis=1)

In [ ]:
# criando listas com os estados
norte = ['Acre', 'Amazonas', 'Rondônia', 'Roraima', 'Pará', 'Amapá', 'Tocantins']
nordeste = ['Maranhão', 'Piauí', 'Rio Grande do Norte', 'Pernambuco', 'Paraíba', 'Alagoas', 'Sergipe', 'Ceará', 'Bahia']
sudeste = ['São Paulo', 'Minas Gerais', 'Espírito Santo', 'Rio de Janeiro']
sul = ['Rio Grande do Sul', 'Paraná', 'Santa Catarina']
centro_oeste = ['Mato Grosso', 'Mato Grosso do Sul', 'Goiás', 'Distrito Federal']

In [ ]:
# criando uma nova coluna para adicionar a região do estado
# usando isin e as listas que criei para indicar que, caso a UF esteja na lista, adicionar a região X
# ex: se o estado está na lista norte, coloque na coluna regiao o valor Norte
df_populacao_uf.loc[df_populacao_uf['UF'].isin(norte), 'regiao'] = 'Norte'
df_populacao_uf.loc[df_populacao_uf['UF'].isin(nordeste), 'regiao'] = 'Nordeste'
df_populacao_uf.loc[df_populacao_uf['UF'].isin(sudeste), 'regiao'] = 'Sudeste'
df_populacao_uf.loc[df_populacao_uf['UF'].isin(sul), 'regiao'] = 'Sul'
df_populacao_uf.loc[df_populacao_uf['UF'].isin(centro_oeste), 'regiao'] = 'Centro Oeste'

In [ ]:
df_populacao_uf.head()

,UF,POPULACAO,regiao
0,Brasil,213421037,NaN
1,Norte,18801282,NaN
2,Rondônia,1751950,Norte
3,Acre,884372,Norte
4,Amazonas,4321616,Norte


Para compreender uma grande quantidade de dados, é necessário resumi-los em
medidas. Entender como essas medidas se relacionam e descrevem os dados é
uma das atividades de Data Analytics. A forma de realizar isso é por meio da
Estatística Descritiva.  
Vamos ao caso!  
O Instituto Aprender encomendou à nossa consultoria a criação de uma amostra
para uma pesquisa com 3500 pessoas. Essa amostra deve ser dividida entre os
estados que possuem a maior população em cada uma das regiões do Brasil.
Você foi o analista selecionado para esse trabalho. Abaixo seguem as análises
solicitadas pelo cliente:


Quantas pessoas serão entrevistadas em cada estado?


In [ ]:
# df com os estados, removendo o total brasil e total por região
linhas_remover = ['Brasil', 'Norte', 'Nordeste', 'Sul', 'Sudeste', 'Centro-Oeste']

# usando o ~ para dizer ao filtro não pegar a linha onde UF = item da lista que criei
df_populacao_uf = df_populacao_uf[~df_populacao_uf['UF'].isin(linhas_remover)].reset_index(drop=True)

In [ ]:
# criando um df para cada região e adicionando os estados conforme o filtro por região
# ex: no df_norte, se a regiao da linha é Norte, o estado será adicionado nesse dataframe
df_norte = df_populacao_uf[df_populacao_uf['regiao'] == 'Norte']
df_nordeste = df_populacao_uf[df_populacao_uf['regiao'] == 'Nordeste']
df_sudeste = df_populacao_uf[df_populacao_uf['regiao'] == 'Sudeste']
df_sul = df_populacao_uf[df_populacao_uf['regiao'] == 'Sul']
df_centroOeste = df_populacao_uf[df_populacao_uf['regiao'] == 'Centro Oeste']

In [ ]:
# pegamos o index que linha cujo a populacao tenha maior valor
# estou fazendo 5 vezes, uma para cada região, ou seja, estou pegando o estado mais populoso de cada região
# depois, nna variável estado_maiorpop_x pego a linha completa desse índice
idx_maior_norte = df_norte['POPULACAO'].idxmax()
estado_maiorpop_norte = df_norte.loc[idx_maior_norte]

idx_maior_nordeste = df_nordeste['POPULACAO'].idxmax()
estado_maiorpop_nordeste = df_nordeste.loc[idx_maior_nordeste]

idx_maior_sudeste = df_sudeste['POPULACAO'].idxmax()
estado_maiorpop_sudeste = df_sudeste.loc[idx_maior_sudeste]

idx_maior_sul = df_sul['POPULACAO'].idxmax()
estado_maiorpop_sul = df_sul.loc[idx_maior_sul]

idx_maior_centroO = df_centroOeste['POPULACAO'].idxmax()
estado_maiorpop_co = df_centroOeste.loc[idx_maior_centroO]

# crio dataframe com os estados de maior população
df_maiores_estados = pd.DataFrame([
    estado_maiorpop_norte,
    estado_maiorpop_nordeste,
    estado_maiorpop_sudeste,
    estado_maiorpop_sul,
    estado_maiorpop_co
])

df_maiores_estados

,UF,POPULACAO,regiao
4,Pará,8711196,Norte
15,Bahia,14870907,Nordeste
19,São Paulo,46081801,Sudeste
20,Paraná,11890517,Sul
25,Goiás,7423629,Centro Oeste


In [ ]:
# .iterrows() permite iterar as linhas de um df para for

# primeiro, somo as 5 populações dos estados mais populosos
populacao_total_5_estados = df_maiores_estados['POPULACAO'].sum()

# aqui, estou fazendo um laço que passa por todas as linha do dataframe dos estados mais populosos
for index, linhas in df_maiores_estados.iterrows():

  # pego o nome de cada estado
  nome_estado = (linhas['UF'])

  # faço a conta pegando a (população do estado / pelo total nos 5 estados mais populosos) * 100
  qtde_pessoas = (linhas['POPULACAO'] / populacao_total_5_estados) * 3500
  qtde_pessoas_inteiro = round(qtde_pessoas) # arredondando o resultado, já que não podemos ter metade de uma pessoa

  print(f'A quantidade de pessoas para entrevista no estado {nome_estado} é {qtde_pessoas_inteiro}')

A quantidade de pessoas para entrevista no estado Pará é 343
A quantidade de pessoas para entrevista no estado Bahia é 585
A quantidade de pessoas para entrevista no estado São Paulo é 1813
A quantidade de pessoas para entrevista no estado Paraná é 468
A quantidade de pessoas para entrevista no estado Goiás é 292




Qual a porcentagem que cada estado representa na pesquisa?

In [ ]:
# aqui, estou fazendo um laço que passa por todas as linha do dataframe dos estados mais populosos
for index, linhas in df_maiores_estados.iterrows():
  nome_estado = (linhas['UF']) # pego o nome de cada estado

  # faço a conta pegando a (população do estado / pelo total nos 5 estados mais populosos) * 3500
  qtde_pessoas = (linhas['POPULACAO'] / populacao_total_5_estados) * 3500
  qtde_pessoas_inteiro = round(qtde_pessoas)

  # para saber a % desse grupo participante da entrevista
  # eu pego a coluna (população do estado / pelo total nos 5 estados mais populosos) * 100
  # não uso a variável que possui a quantidade de pessoas para não dar diferença na porcentagem e alterar o resultado
  proporcao = round((linhas['POPULACAO'] / populacao_total_5_estados) * 100, 2)
  print(f'A quantidade de pessoas para entrevista no estado {nome_estado} é {qtde_pessoas_inteiro}. A propoção dessa população na pesquisa é {proporcao}%')

A quantidade de pessoas para entrevista no estado Pará é 343. A propoção dessa população na pesquisa é 9.79%
A quantidade de pessoas para entrevista no estado Bahia é 585. A propoção dessa população na pesquisa é 16.71%
A quantidade de pessoas para entrevista no estado São Paulo é 1813. A propoção dessa população na pesquisa é 51.79%
A quantidade de pessoas para entrevista no estado Paraná é 468. A propoção dessa população na pesquisa é 13.36%
A quantidade de pessoas para entrevista no estado Goiás é 292. A propoção dessa população na pesquisa é 8.34%


Qual é a média de pessoas que serão entrevistadas nos estados?

In [ ]:
numero_de_estados = len(df_maiores_estados)

media_entrevistados = 3500 / numero_de_estados

print(f'A média de entrevistados por estado é: {media_entrevistados}')

A média de entrevistados por estado é: 700.0


Qual deve ser a margem de erro da pesquisa tendo um nível de confiança
de 95% e p = 0,5?

Fórmula para o cálculo de margem de erro:
$$e = Z \times \sqrt{\frac{p \times (1 - p)}{n}}$$

onde:
* e = erro
* Z = nível de confiança (95% ou 1,96) = 1,96 representa o número de desvios padrão (valor Z) necessário para cobrir 95% dos dados em uma distribuição normal.
* p = proporção/variabilidade (0,5)
* n = tamanho da amostra / total de pessoas que serão entrevistadas (3500)

In [ ]:
z = 1.96 # confiança
p = 0.5 # variabilidade
n = 3500 # tamanho da amostra

# multiplixo z pela raiz quadrada, aqui é ** 0.5 da conta de p * (1-p) dividido por n
e = z * ((p * (1 - p)) / n) ** 0.5

print(f'A margem de erro é {(e*100):.2f}%')

A margem de erro é 1.66%
